# Live demo: our best model on a test pair

Late fusion of the fine-tuned BERT (title) and the MLP on frozen ViT-B/16 features (photo),
94.7 % top-1 on the test split. Both branches run live on the raw photo and title.

Needs the data and the seed-0 outputs of `just train-image-mlp`, `just train-bert-finetune` and
`just train-late-fusion text.method=text_bert_ft name=late_calibrated_ft sum_name=null`.
Register the kernel once with `just kernel`, then pick the `recipe-fusion` kernel.

In [ ]:
import random
from pathlib import Path

from IPython.display import display
from PIL import Image

from recipe_fusion.data.upmc import read_split
from recipe_fusion.demo import data_dir, load_best, top_k

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
model = load_best(ROOT)
test = read_split(data_dir(ROOT), "test")
print(f"ready on {model.device}, {len(test):,} test pairs")

`INDEX`: `None` for a random test pair, or its position in the test file (0 to 22,715).

In [ ]:
INDEX = None

index = random.randrange(len(test)) if INDEX is None else INDEX
sample = test[index]
image = Image.open(sample.image_path(data_dir(ROOT)))
display(image.resize((320, round(320 * image.height / image.width))))
print(f"#{index}  title: {sample.text}")

In [ ]:
for label, p in top_k(model.predict(image, sample.text), model.classes):
    mark = "✅" if label == sample.label else "  "
    print(f"{mark} {label:<24} {p:6.1%}  {'█' * round(30 * p)}")
print(f"\ntruth: {sample.label}")